# Continual Learning: Training Neural Networks That Do Not Forget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/continual_learning.ipynb)

Companion notebook to [Continual Learning: Training Neural Networks That Do Not Forget](https://sesen.ai/blog/continual-learning-catastrophic-forgetting-ewc).

We split MNIST into five two-digit tasks and train one network on them in sequence. Naive training **forgets** the first task; **EWC** and **replay** fix it. We measure exactly how much.

## Setup

In [ ]:
# Uncomment on Colab
# !pip install torch torchvision matplotlib numpy

In [ ]:
import copy, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
TASKS = [(0,1),(2,3),(4,5),(6,7),(8,9)]
print('device', DEVICE)

## Split-MNIST: five two-digit tasks

Each task is a binary problem (e.g. 0 vs 1). The network has 10 outputs; at test time for a task we score only that task's two units (task-incremental, so chance is 50%).

In [ ]:
tf = transforms.ToTensor()
tr = datasets.MNIST('./data', train=True, download=True, transform=tf)
te = datasets.MNIST('./data', train=False, download=True, transform=tf)
Xtr, ytr = tr.data.float().view(-1,784)/255., tr.targets
Xte, yte = te.data.float().view(-1,784)/255., te.targets
tasks_tr = [((ytr==a)|(ytr==b)) for a,b in TASKS]
tasks_tr = [(Xtr[m], ytr[m]) for m in tasks_tr]
tasks_te = [((yte==a)|(yte==b)) for a,b in TASKS]
tasks_te = [(Xte[m], yte[m]) for m in tasks_te]
print('task sizes:', [len(x) for x,_ in tasks_tr])

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784,256), nn.ReLU(),
                                 nn.Linear(256,256), nn.ReLU(), nn.Linear(256,10))
    def forward(self, x): return self.net(x)

def task_logits(logits, t):
    a, b = TASKS[t]; return logits[:, [a, b]]

@torch.no_grad()
def evaluate(model):
    model.eval(); accs = []
    for t,(X,y) in enumerate(tasks_te):
        a,b = TASKS[t]
        pred = task_logits(model(X.to(DEVICE)), t).argmax(1)
        tgt = (y==b).long().to(DEVICE)
        accs.append((pred==tgt).float().mean().item())
    model.train(); return accs

## Fisher information for EWC

The diagonal Fisher (mean squared gradient of the loss) measures how important each weight was for a finished task. EWC penalises moving high-Fisher weights.

In [ ]:
def fisher(model, X, y, t, n=512):
    f = {k: torch.zeros_like(p) for k,p in model.named_parameters()}
    a,b = TASKS[t]
    for k in torch.randperm(len(X))[:n]:
        model.zero_grad()
        tgt = (y[k:k+1]==b).long().to(DEVICE)
        loss = F.cross_entropy(task_logits(model(X[k:k+1].to(DEVICE)), t), tgt)
        loss.backward()
        for k_,p in model.named_parameters():
            if p.grad is not None: f[k_] += p.grad.detach()**2
    return {k_: fi/n for k_,fi in f.items()}

## One training loop, three strategies

`method` selects naive SGD, EWC (Fisher penalty), or replay (memory buffer). After each task we record accuracy on every task to build the accuracy matrix.

In [ ]:
def train(method='naive', ewc_lambda=2000., buf_per_task=200, epochs=3, bs=128):
    model = MLP().to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    fishers, stars, bufX, bufY, bufT, mat = [], [], [], [], [], []
    for t,(X,y) in enumerate(tasks_tr):
        a,b = TASKS[t]
        for _ in range(epochs):
            idx = torch.randperm(len(X))
            for i in range(0, len(X), bs):
                j = idx[i:i+bs]; xb = X[j].to(DEVICE)
                loss = F.cross_entropy(task_logits(model(xb), t), (y[j]==b).long().to(DEVICE))
                if method=='ewc':
                    for fsh, st in zip(fishers, stars):
                        for k_,p in model.named_parameters():
                            loss = loss + (ewc_lambda/2)*(fsh[k_]*(p-st[k_])**2).sum()
                if method=='replay' and bufX:
                    bi = torch.randint(0, len(bufX), (min(bs,len(bufX)),))
                    rx = torch.stack([bufX[i] for i in bi]).to(DEVICE)
                    rt = torch.tensor([bufT[i] for i in bi]); ry = torch.stack([bufY[i] for i in bi])
                    rl = 0.
                    for pt in rt.unique():
                        mm = rt==pt; aa,bb = TASKS[int(pt)]
                        rl = rl + F.cross_entropy(task_logits(model(rx[mm]), int(pt)), (ry[mm]==bb).long().to(DEVICE))
                    loss = loss + rl/len(rt.unique())
                opt.zero_grad(); loss.backward(); opt.step()
        if method=='ewc':
            fishers.append(fisher(model, X, y, t))
            stars.append({k_:p.detach().clone() for k_,p in model.named_parameters()})
        if method=='replay':
            for i in torch.randperm(len(X))[:buf_per_task]:
                bufX.append(X[i]); bufY.append(y[i]); bufT.append(t)
        mat.append(evaluate(model))
    return np.array(mat)

In [ ]:
results = {}
for m in ['naive','ewc','replay']:
    M = train(m); results[m] = M
    final = M[-1]; peak = np.array([M[t:,t].max() for t in range(5)])
    forget = (peak-final)[:-1].mean()
    print(f'{m:7s} avg_acc {final.mean():.3f} | task1 {final[0]:.3f} | forgetting {forget:.3f}', flush=True)

## Visualising forgetting

In [ ]:
colors = {'naive':'#dc2626','ewc':'#2563eb','replay':'#059669'}
fig, ax = plt.subplots(figsize=(8,5))
for m in ['naive','ewc','replay']:
    ax.plot(range(1,6), results[m][:,0], 'o-', color=colors[m], lw=2.4, label=m)
ax.axhline(0.5, ls=':', color='gray'); ax.set_ylim(0.45,1.02)
ax.set_xlabel('tasks learned'); ax.set_ylabel('accuracy on Task 1 (0/1)')
ax.set_title('Catastrophic forgetting of the first task'); ax.legend(); ax.grid(alpha=.3)
plt.show()

## What to remember

- **Catastrophic forgetting** is structural: shared weights get overwritten by new-task gradients.
- **EWC** anchors high-Fisher weights near their old values (Bayesian updating in disguise).
- **Replay** keeps a little old data and is the simplest strong baseline.
- For LLMs, **parameter isolation** (LoRA adapters on a frozen base) is the dominant fix, often plus replay.

## Exercises

1. **Sweep EWC's lambda** over {0, 200, 2000, 20000}. Plot the stability-plasticity trade-off: forgetting versus new-task accuracy.
2. **Shrink the replay buffer** to 50, 20, 5 examples per task. How small can it get before forgetting returns?
3. **Class-incremental mode.** Evaluate over all 10 outputs with no task identity. Watch EWC struggle and replay's lead grow.
4. **Combine EWC + replay.** Does the combination beat either alone?
5. **Permuted MNIST.** Replace split tasks with fixed random pixel permutations (the original EWC benchmark) and compare.